In [5]:
"""Cover fractions and 20% labels for every 1000, 500 and 250 px tile.

Reads the "tiles" and "rhodo" layers from labelling.gpkg and writes
tiles_1000px.gpkg, tiles_500px.gpkg and tiles_250px.gpkg.

Usage: python cover_fractions.py labelling.gpkg
Close QGIS (or save all edits) before running.
"""
import sys

import geopandas as gpd
from shapely.geometry import box

LABELLING_PATH = "../Grids/labelling.gpkg"
COVER_THRESHOLD = 0.20                   # cover >= 20% -> positive
LABELLED_STATUSES = ["done", "unsure"]   # "skip" and unlabelled tiles are left out
SIZES = [(1000, 1), (500, 2), (250, 4)]  # (tile size in px, splits per side)

tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
rhododendron_outlines = gpd.read_file(LABELLING_PATH, layer="rhododendron_outlines")
rhododendron_outlines["geometry"] = rhododendron_outlines.make_valid()
rhodo_area = rhododendron_outlines.union_all()           # merge overlaps so no area counts twice

labelled = tiles[tiles["status"].isin(LABELLED_STATUSES)]
print(f"{len(labelled)} labelled tiles, {(tiles['status'] == 'skip').sum()} skipped, "
      f"{tiles['status'].isna().sum()} with no status")

keep = [c for c in tiles.columns if c != "geometry"]
rows = {size: [] for size, _ in SIZES}
for tile in labelled.itertuples(index=False):
    min_x, min_y, max_x, max_y = tile.geometry.bounds
    rhodo_in_tile = rhodo_area.intersection(tile.geometry)   # clip once per tile
    for size, splits in SIZES:
        step_x, step_y = (max_x - min_x) / splits, (max_y - min_y) / splits
        for sub_row in range(splits):                         # row 0 = top
            for sub_col in range(splits):
                sub = box(min_x + sub_col * step_x, max_y - (sub_row + 1) * step_y,
                          min_x + (sub_col + 1) * step_x, max_y - sub_row * step_y)
                cover = rhodo_in_tile.intersection(sub).area / sub.area
                label = int(cover >= COVER_THRESHOLD)
                rows[size].append({**{c: getattr(tile, c) for c in keep},
                                   "tile_size_px": size, "sub_row": sub_row, "sub_col": sub_col,
                                   "cover_fraction": round(cover, 4), "label": label,
                                   "class": "positive" if label else "negative",
                                   "geometry": sub})

for size, _ in SIZES:
    out = gpd.GeoDataFrame(rows[size], geometry="geometry", crs=tiles.crs)
    path = f"../Grids/tiles_{size}px.gpkg"
    out.to_file(path, layer=f"tiles_{size}px", driver="GPKG")
    print(f"{path}: {len(out)} tiles, {out['label'].sum()} positive "
          f"({out['label'].mean():.1%}), mean cover {out['cover_fraction'].mean():.3f}")

2691 labelled tiles, 0 skipped, 2490 with no status
../Grids/tiles_1000px.gpkg: 2691 tiles, 244 positive (9.1%), mean cover 0.059
../Grids/tiles_500px.gpkg: 10764 tiles, 925 positive (8.6%), mean cover 0.059
../Grids/tiles_250px.gpkg: 43056 tiles, 3241 positive (7.5%), mean cover 0.059


In [ ]:
import pandas as pd
fold_of_block = pd.read_csv("../Grids/folds_KD02.csv").set_index("block")["fold"]
for s in [1000, 500, 250]:
    g = gpd.read_file(f"../Grids/tiles_{s}px.gpkg")
    g["fold"] = g["block"].map(fold_of_block)
    g.to_file(f"../Grids/tiles_{s}px.gpkg", layer=f"tiles_{s}px", driver="GPKG")
    print(s, "px: tiles with no fold:", int((g["fold"].isna() & (g["forest"] == "KD02") & (g["role"] == "train")).sum()))

In [1]:
import geopandas as gpd
t = gpd.read_file("../Grids/tiles_500px.gpkg")
t = t[(t["forest"] == "KD02") & (t["role"] == "train")]
for name, part in [("tiles 1-500", t[t["order"] <= 500]), ("tiles 501-1000", t[t["order"] > 500])]:
    done = part[part["status"] == "done"]
    print(f"{name}: {done['tile_id'].nunique()} done, {part[part['status'] == 'unsure']['tile_id'].nunique()} unsure, "
          f"{int(done['label'].sum())} positive 500 px tiles in {done[done['label'] == 1]['block'].nunique()} blocks")

tiles 1-500: 420 done, 80 unsure, 171 positive 500 px tiles in 10 blocks
tiles 501-1000: 435 done, 65 unsure, 206 positive 500 px tiles in 10 blocks


In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

TIFF_PATH = "../LargeImages/KD02-1-20241203-7_IXM150_mos.tif"
GRID_DIR = Path("../Grids")
OUT_DIR = Path("../ImageFiles/InitialComparison")
SPLITS = {1000: 1, 500: 2, 250: 4}       # sub-tiles per side

# Stage A tiles: KD02 training blocks, "done" only, with a fold
parents = gpd.read_file(GRID_DIR / "tiles_1000px.gpkg")
parents = parents[(parents["forest"] == "KD02") & (parents["role"] == "train")
                  & (parents["status"] == "done") & parents["fold"].notna()]
print(f"{len(parents)} tiles")

written, skipped, problems = 0, 0, []
with rasterio.open(TIFF_PATH) as ds:
    for tile in tqdm(parents.itertuples(), total=len(parents)):
        names = {(size, r, c): OUT_DIR / str(size) / f"{tile.tile_id}_{size}_{r}_{c}.png"
                 for size, n in SPLITS.items() for r in range(n) for c in range(n)}
        if all(p.exists() for p in names.values()):        # already cut earlier
            skipped += 1
            continue
        window = from_bounds(*tile.geometry.bounds, transform=ds.transform).round_offsets().round_lengths()
        rgb = ds.read([1, 2, 3], window=window, boundless=True, fill_value=0)   # bands 1-3 only
        rgb = np.moveaxis(rgb, 0, -1)                                            # (rows, cols, 3)
        if rgb.shape[:2] != (1000, 1000):
            problems.append((tile.tile_id, rgb.shape))
            continue
        for (size, r, c), path in names.items():            # row 0 = top, as in the GeoPackages
            Image.fromarray(rgb[r * size:(r + 1) * size, c * size:(c + 1) * size]).save(path)
            written += 1

print(f"Saved {written} new images; {skipped} tiles already cut")
if problems:
    print(f"{len(problems)} tiles not 1000 x 1000 px, skipped:", problems[:10])

In [2]:
import geopandas as gpd
g = gpd.read_file("../Grids/tiles_500px.gpkg")       # in Colab: f"{PROJECT}/Grids/tiles_500px.gpkg"
use = g[(g["forest"] == "KD02") & (g["role"] == "train") & (g["status"] == "done") & g["fold"].notna()]
print(len(use), "usable 500 px tiles; highest order:", int(use["order"].max()))

3420 usable 500 px tiles; highest order: 1000


In [8]:
import geopandas as gpd
t = gpd.read_file("../Grids/tiles_1000px.gpkg")
print(t.groupby(["forest", "role", "status"]).size())
test = t[t["role"] == "test"]
print("\ntest tiles per block:\n", test.groupby("block").size())
print("test tiles with no condition:", test["condition"].isna().sum())
print(test.groupby(["forest", "condition"]).size())

forest  role   status
KD02    test   done      343
               unsure     33
        train  done      855
               unsure    145
KD06    test   done      283
               unsure     32
        train  done      930
               unsure     70
dtype: int64

test tiles per block:
 block
KD02_2002    169
KD02_4002    207
KD06_1002    169
KD06_2005    146
dtype: int64
test tiles with no condition: 0
forest  condition     
KD02    built              12
        conifer            35
        deciduous         118
        field              74
        mixed_woodland     85
        open_woodland      52
KD06    conifer            39
        deciduous         117
        field              29
        mixed_woodland     57
        open_woodland      73
dtype: int64


In [9]:
import pandas as pd, geopandas as gpd
from sklearn.model_selection import StratifiedGroupKFold

KD06_FOLD_SEED = 20261119      # record this in your methods
SIZES = [1000, 500, 250]

# KD06: 3 folds by block, from 1000 px training tiles marked "done"
t = gpd.read_file("../Grids/tiles_1000px.gpkg")
p = t[(t["forest"] == "KD06") & (t["role"] == "train") & (t["status"] == "done")]
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=KD06_FOLD_SEED)
kd06 = {}
for fold, (_, idx) in enumerate(cv.split(p, p["label"], groups=p["block"])):
    for b in p.iloc[idx]["block"].unique():
        kd06[b] = fold
pd.DataFrame({"block": list(kd06), "fold": list(kd06.values())}).to_csv("../Grids/folds_KD06.csv", index=False)

# Combine with KD02's saved folds (block names are unique per forest)
fold_of_block = {**pd.read_csv("../Grids/folds_KD02.csv").set_index("block")["fold"].to_dict(), **kd06}
for s in SIZES:
    g = gpd.read_file(f"../Grids/tiles_{s}px.gpkg")
    g["fold"] = g["block"].map(fold_of_block)          # test-block tiles stay blank, as intended
    g.to_file(f"../Grids/tiles_{s}px.gpkg", layer=f"tiles_{s}px", driver="GPKG")
    tr = g[(g["role"] == "train") & (g["status"] == "done")]
    print(f"\n{s} px: training tiles with no fold:", int(tr["fold"].isna().sum()))
    print(tr.groupby(["forest", "fold"]).agg(tiles=("label", "size"), positives=("label", "sum")))


1000 px: training tiles with no fold: 0
             tiles  positives
forest fold                  
KD02   0.0     148         25
       1.0     140          8
       2.0     201         14
       3.0     168         16
       4.0     198         42
KD06   0.0     311         27
       1.0     299         28
       2.0     320         25

500 px: training tiles with no fold: 0
             tiles  positives
forest fold                  
KD02   0.0     592         88
       1.0     560         29
       2.0     804         46
       3.0     672         57
       4.0     792        157
KD06   0.0    1244        113
       1.0    1196         97
       2.0    1280         92

250 px: training tiles with no fold: 0
             tiles  positives
forest fold                  
KD02   0.0    2368        306
       1.0    2240         96
       2.0    3216        145
       3.0    2688        197
       4.0    3168        552
KD06   0.0    4976        402
       1.0    4784        353
       2.

In [10]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

TIFF_PATHS = {"KD02": "../LargeImages/KD02-1-20241203-7_IXM150_mos.tif",
              "KD06": "../LargeImages/KD06-1-20241203-7_IXM150_mos.tif"}
GRID_DIR = Path("../Grids")
SIZE = 500                                              # stage B tile size
OUT_DIR = Path("../ImageFiles/FinalTiles") / str(SIZE)  # same layout as before: <image_dir>/500/<file>.png
OUT_DIR.mkdir(parents=True, exist_ok=True)
N = 1000 // SIZE                                        # sub-tiles per side (2)

all_tiles = gpd.read_file(GRID_DIR / "tiles_1000px.gpkg")
is_train = (all_tiles["role"] == "train") & (all_tiles["status"] == "done") & all_tiles["fold"].notna()
is_test = (all_tiles["role"] == "test") & all_tiles["status"].isin(["done", "unsure"])   # test "unsure" kept

for forest, tiff in TIFF_PATHS.items():
    parents = all_tiles[(all_tiles["forest"] == forest) & (is_train | is_test)]
    print(f"\n{forest}: {len(parents)} tiles ({int((parents['role'] == 'train').sum())} train, "
          f"{int((parents['role'] == 'test').sum())} test)")
    written, skipped, problems = 0, 0, []
    with rasterio.open(tiff) as ds:
        for tile in tqdm(parents.itertuples(), total=len(parents), desc=forest):
            names = {(r, c): OUT_DIR / f"{tile.tile_id}_{SIZE}_{r}_{c}.png" for r in range(N) for c in range(N)}
            if all(p.exists() for p in names.values()):        # already cut
                skipped += 1
                continue
            window = from_bounds(*tile.geometry.bounds, transform=ds.transform).round_offsets().round_lengths()
            rgb = np.moveaxis(ds.read([1, 2, 3], window=window, boundless=True, fill_value=0), 0, -1)
            if rgb.shape[:2] != (1000, 1000):
                problems.append((tile.tile_id, rgb.shape))
                continue
            for (r, c), path in names.items():                  # row 0 = top, as in the GeoPackages
                Image.fromarray(rgb[r * SIZE:(r + 1) * SIZE, c * SIZE:(c + 1) * SIZE]).save(path)
                written += 1
    print(f"{forest}: saved {written} images; {skipped} tiles already cut")
    if problems:
        print(f"{forest}: {len(problems)} tiles not 1000 x 1000 px, skipped:", problems[:10])

print(f"\nTotal images in {OUT_DIR}: {len(list(OUT_DIR.glob('*.png')))}")

/home/cyan/miniconda3/envs/RhodaiTF/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



KD02: 1231 tiles (855 train, 376 test)


KD02: 100%|██████████| 1231/1231 [17:02<00:00,  1.20it/s]


KD02: saved 4924 images; 0 tiles already cut

KD06: 1245 tiles (930 train, 315 test)


KD06: 100%|██████████| 1245/1245 [25:07<00:00,  1.21s/it]

KD06: saved 4980 images; 0 tiles already cut

Total images in ../ImageFiles/FinalTiles/500: 9904


In [1]:
import geopandas as gpd
lab = gpd.read_file("../Grids/labelling.gpkg", layer="tiles")
test = lab[lab["role"] == "test"]
print(test.groupby("forest")["in_shadow"].value_counts(dropna=False))
test[["tile_id", "forest", "in_shadow"]].to_csv("../Grids/test_shadow.csv", index=False)

forest  in_shadow
KD02    0.0          337
        1.0           39
KD06    0.0          269
        1.0           46
Name: count, dtype: int64
